# Feature selection
**Feature selection** is the process of identifying and keeping only the most relevant input variables (features) in a dataset to build efficient, accurate, and interpretable machine learning models. It reduces noise, speeds up training, and helps avoid overfitting

# why feature selection matters?
1. **Avoid curse of dimensinality**-High-dimensional data can degrade performance (due to sparsity)
2. High dimensional data cause **computation complexity**
3. **Interpretability issue*

# Types of feature selection
1. **Filter-based method**
2. **Wrapper method**
3. **Embedded method**
4. **Hybrid technique**

# Filter-Based Method
Feature‑based methods in feature selection evaluate each feature individually using statistical or mathematical criteria, without relying on a specific machine learning model. They act like a quick filter to remove irrelevant or redundant inputs before training. Common techniques include correlation tests, chi‑square tests, mutual information, and variance thresholds. These methods are fast, simple, and scalable for large datasets, but they don’t account for interactions between features, which can sometimes limit their effectiveness

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
df=pd.read_csv(r"C:\Users\Sonu\OneDrive\Desktop\machine learning\feature selction\train.csv")

In [6]:
df.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1,STANDING
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1,STANDING


In [11]:
df.info()

# print(df.describe())

print(df["Activity"].value_counts())

print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7352 entries, 0 to 7351
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 31.6+ MB
Activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64
(7352, 563)


**we have 563 columns and around 7000 rows**
firstly we apply logistic regression and then we do feature selection and then compare the results

In [26]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# seprate features and target
x=df.drop('Activity',axis=1)
y=df['Activity']

# label encoding
le=LabelEncoder()
y=le.fit_transform(y)

# train test split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [27]:
print(x_train.shape)
print(x_test.shape)

(5881, 562)
(1471, 562)


# Applying logistic Regression

In [28]:
log_reg=LogisticRegression(max_iter=1000)
log_reg.fit(x_train,y_train)

# make prediction on test data
y_pred=log_reg.predict(x_test)

# calculate r2 score
r2_score=accuracy_score(y_test,y_pred)
print("Accuracy score:",r2_score)

Accuracy score: 0.981645139360979


When we have 563 columns then we accuracy of 98%
Now,we drop 463 columns, we take this with 100 columns we get accuracy almost same

# check duplicate column

In [29]:
import pandas as pd

def get_duplicate_columns(df):
    duplicate_columns = {}
    seen_columns = {}

    for column in df.columns:
        current_column = df[column]

        # Convert column data to bytes for hashing
        try:
            current_column_hash = current_column.values.tobytes()
        except AttributeError:
            current_column_hash = current_column.to_string().encode()

        # Check if we've seen this column before
        if current_column_hash in seen_columns:
            first_col = seen_columns[current_column_hash]
            if first_col in duplicate_columns:
                duplicate_columns[first_col].append(column)
            else:
                duplicate_columns[first_col] = [column]
        else:
            seen_columns[current_column_hash] = column

    return duplicate_columns

duplicates = get_duplicate_columns(x_train)
print("Duplicate columns:", duplicates)


Duplicate columns: {'tBodyAccMag-mean()': ['tBodyAccMag-sma()', 'tGravityAccMag-mean()', 'tGravityAccMag-sma()'], 'tBodyAccMag-std()': ['tGravityAccMag-std()'], 'tBodyAccMag-mad()': ['tGravityAccMag-mad()'], 'tBodyAccMag-max()': ['tGravityAccMag-max()'], 'tBodyAccMag-min()': ['tGravityAccMag-min()'], 'tBodyAccMag-energy()': ['tGravityAccMag-energy()'], 'tBodyAccMag-iqr()': ['tGravityAccMag-iqr()'], 'tBodyAccMag-entropy()': ['tGravityAccMag-entropy()'], 'tBodyAccMag-arCoeff()1': ['tGravityAccMag-arCoeff()1'], 'tBodyAccMag-arCoeff()2': ['tGravityAccMag-arCoeff()2'], 'tBodyAccMag-arCoeff()3': ['tGravityAccMag-arCoeff()3'], 'tBodyAccMag-arCoeff()4': ['tGravityAccMag-arCoeff()4'], 'tBodyAccJerkMag-mean()': ['tBodyAccJerkMag-sma()'], 'tBodyGyroMag-mean()': ['tBodyGyroMag-sma()'], 'tBodyGyroJerkMag-mean()': ['tBodyGyroJerkMag-sma()'], 'fBodyAccMag-mean()': ['fBodyAccMag-sma()'], 'fBodyBodyAccJerkMag-mean()': ['fBodyBodyAccJerkMag-sma()'], 'fBodyBodyGyroMag-mean()': ['fBodyBodyGyroMag-sma()'],

In [30]:
duplicate_columns = get_duplicate_columns(x_train)

In [31]:
for one_list in duplicates.values():
    x_train.drop(one_list,axis=1,inplace=True)
    x_test.drop(one_list,axis=1,inplace=True)

In [32]:
print(x_train.shape)
print(x_test.shape)

(5881, 541)
(1471, 541)


**Here we almost drop 20 columns beacuse these columns just duplicate of other or copy paste value**
# Step 1. Variance Threshold
It keeps only features whose variance is greater than a chosen threshold. Constant or near‑constant columns are eliminated, reducing noise and dimensionality.

In [33]:
from sklearn.feature_selection import VarianceThreshold
sel=VarianceThreshold(threshold=0.05)
sel.fit(x_train)

,threshold,0.05


In [37]:
sum(sel.get_support())
columns=x_train.columns[sel.get_support()]
print(columns)

Index(['tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z',
       'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z',
       'tBodyAcc-max()-X', 'tBodyAcc-max()-Y', 'tBodyAcc-max()-Z',
       'tBodyAcc-min()-X',
       ...
       'fBodyBodyGyroJerkMag-skewness()', 'fBodyBodyGyroJerkMag-kurtosis()',
       'angle(tBodyAccMean,gravity)', 'angle(tBodyAccJerkMean),gravityMean)',
       'angle(tBodyGyroMean,gravityMean)',
       'angle(tBodyGyroJerkMean,gravityMean)', 'angle(X,gravityMean)',
       'angle(Y,gravityMean)', 'angle(Z,gravityMean)', 'subject'],
      dtype='object', length=350)


In [38]:
x_train=sel.transform(x_train)
x_test=sel.transform(x_test)

x_train=pd.DataFrame(x_train,columns=columns)
x_test=pd.DataFrame(x_test,columns=columns)

In [39]:
print(x_train.shape)
print(x_test.shape)

(5881, 350)
(1471, 350)


# Step2-correlaton
A correlation matrix is a table that shows the strength and direction of relationships between multiple variables at once, using correlation coefficients that range from -1 to +1. It’s widely used in data analysis, machine learning, finance, and research to quickly identify patterns, dependencies, or redundancies among variables.

In [40]:
corr_matrix=x_train.corr()

In [41]:
# get columns names
columns=corr_matrix.columns

# create empity list to store columns to drop
columns_to_drop = []

# loop through the columns of the correlation matrix
for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        if corr_matrix.loc[columns[i], columns[j]] > 0.95:
            # colname = columns[i]
            columns_to_drop.append(columns[j])

print("Columns to drop due to high correlation:", len(set(columns_to_drop)))

Columns to drop due to high correlation: 197


**we clearly see there we can drop 197 more column because high coorelation and multicoolinearity**

In [48]:
# x_train.drop(columns=columns_to_drop, axis=1, inplace=True)
x_train.shape
# x_test.drop(columns=columns_to_drop, axis=1, inplace=True)
x_test.shape

(1471, 153)

**Now,we have only 153 columns left, intially we have 563 columns**

#  Apply ANOVA
ANOVA (Analysis of Variance) is a statistical test used to compare the means of three or more groups to see if at least one group differs significantly. It works by analyzing the variability between groups versus within groups and is widely used in experiments, medicine, business, and social sciences. 


In [ ]:
from sklearn.feature_selection import f_classif, SelectKBest
sel=SelectKBest(f_classif,k=100).fit(x_train,y_train)
x_train.columns[sel.get_support()]  # these are the 100 best features based on ANOVA F-value between label/feature for classification tasks

Index(['tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z',
       'tBodyAcc-max()-Z', 'tBodyAcc-min()-X', 'tBodyAcc-min()-Y',
       'tBodyAcc-min()-Z', 'tBodyAcc-entropy()-X', 'tBodyAcc-entropy()-Y',
       'tBodyAcc-entropy()-Z', 'tBodyAcc-arCoeff()-X,1',
       'tBodyAcc-arCoeff()-X,2', 'tBodyAcc-arCoeff()-X,3',
       'tBodyAcc-arCoeff()-Y,1', 'tBodyAcc-arCoeff()-Z,1',
       'tBodyAcc-correlation()-X,Y', 'tBodyAcc-correlation()-Y,Z',
       'tGravityAcc-mean()-X', 'tGravityAcc-mean()-Y', 'tGravityAcc-mean()-Z',
       'tGravityAcc-sma()', 'tGravityAcc-energy()-Y', 'tGravityAcc-energy()-Z',
       'tGravityAcc-entropy()-X', 'tGravityAcc-entropy()-Y',
       'tGravityAcc-arCoeff()-Y,1', 'tGravityAcc-arCoeff()-Y,2',
       'tGravityAcc-arCoeff()-Z,1', 'tGravityAcc-arCoeff()-Z,2',
       'tGravityAcc-correlation()-Y,Z', 'tBodyAccJerk-std()-Z',
       'tBodyAccJerk-min()-X', 'tBodyAccJerk-min()-Y', 'tBodyAccJerk-min()-Z',
       'tBodyAccJerk-entropy()-X', 'tBodyAccJerk-arCoeff

In [50]:
columns=x_train.columns[sel.get_support()]

In [51]:
x_train=sel.transform(x_train)
x_test=sel.transform(x_test)

x_train=pd.DataFrame(x_train,columns=columns)
x_test=pd.DataFrame(x_test,columns=columns)

In [ ]:
print(x_train.shape)
print(x_train.shape)  # Now,we have only 100 best columns 

(5881, 100)
(5881, 100)


In [55]:
log_reg=LogisticRegression(max_iter=1000)
log_reg.fit(x_train,y_train)

y_pred=log_reg.predict(x_test)

r2_score=accuracy_score(y_test,y_pred)
print(r2_score)

0.9687287559483345


**Here,We clearly see that after removing 463  columns our accuracy diff comes only 1.2%. so,we conclude that remaing 463 columns not much contribute in prediction**